# **AI로 웹데이터 수집 및 시각화**



---



## **1-4.동적 크롤링**(**Visual Studio Code 사용**)

- 공식페이지: https://www.selenium.dev/
- 참고: https://wikidocs.net/198942
- beautifulsoup 사용법 : https://wikidocs.net/85739
    - **select_one** 은 찾은 html 중 가장 첫번째 html 을 가져오고
    - **select** 는 찾은 모든 html 을 리스트 형태로 반환
- 크롬 브라우저 필요
- ※ 동적 크롤링은 PC에서 실행하세요 (IDLE 또는 Visual Studio Code)


* **라이브러리 설치하기**

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
!pip install requests beautifulsoup4 selenium chromedriver-autoinstaller

- **한글 폰트 지정하기**

In [ ]:
# 코랩에서 한글 폰트 종류와 이름이 win과 다를 수 있다!!!
# 코랩: NanumGothic, 윈도우: Malgun Gothic
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Malgun Gothic',
                     'font.size': 12,
                     'figure.figsize': (6, 4),
                     'axes.unicode_minus':  False }) # 폰트 설정

In [ ]:
import selenium
selenium.__version__

### 1️⃣ 웹 드라이브 테스트

In [ ]:
from selenium import webdriver

# 드라이버 초기화
driver = webdriver.Chrome()

# 웹페이지로 이동
driver.get('https://www.weather.go.kr')

# 브라우저 탭 닫기
driver.close()

# 브라우저 종료하기 (탭 모두 종료)
driver.quit()

* **Target 웹 페이지**

In [ ]:
# ============================================================
# 동적 크롤링 — 기상청 시간별 예보 수집하기 (Selenium)
# 대상 페이지 : https://www.weather.go.kr/w/weather/forecast/short-term.do
# ============================================================
import os
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import StaleElementReferenceException

URL = 'https://www.weather.go.kr/w/weather/forecast/short-term.do'
FILE = os.path.abspath('시간별예보.csv')   # 저장 위치를 '절대경로'로 고정한다

# 표로 만들 항목 — 여기만 고치면 항목을 자유롭게 넣고 뺄 수 있다.
COLUMNS = ['시각', '날씨', '기온', '체감온도', '강수확률', '습도']

ITEM = '#digital-forecast ul.item.s-item'   # 시간대 한 칸


# [함수] 예보가 다 그려졌는지 확인한다 -----------------------------
#   ※ 페이지가 "다시 그려지는 중"에 요소를 읽으면
#      StaleElementReferenceException(낡은 참조 오류)이 난다.
#      → 그럴 땐 False를 돌려주고 다음 번에 다시 확인하면 된다.
def 예보_로딩됨(d):
    try:
        return any('시각' in (li.get_attribute('textContent') or '')
                   for li in d.find_elements(By.CSS_SELECTOR, ITEM + ' li'))
    except StaleElementReferenceException:
        return False


# 1) 크롬을 자동 실행하고 단기예보 페이지 열기 ---------------------
driver = webdriver.Chrome()

try:
    driver.get(URL)

    # 2) 시간별 예보가 "로딩될 때까지" 기다리기 (동적 크롤링의 핵심!) --
    #    requests로 받으면 '로딩중...'만 보이지만,
    #    Selenium은 자바스크립트 실행이 끝난 화면을 읽을 수 있다.
    #    ※ 빈 표 껍데기가 먼저 그려지므로, '시각' 글자가 들어올 때까지 기다린다.
    wait = WebDriverWait(driver, 30)          # 인터넷이 느려도 되도록 30초로 넉넉하게
    wait.until(예보_로딩됨)
    print('시간별 예보 로딩 완료!')

    # 3) ★ 완성된 화면을 통째로 복사해 둔다 ★ -------------------------
    #    브라우저에서 한 칸씩 읽으면 그 사이에 화면이 바뀌어 낡은 참조 오류가 난다.
    #    html 문자열로 한 번에 떠 두면 그런 일이 없다.
    html = driver.page_source

finally:
    driver.quit()          # 오류가 나도 크롬은 반드시 닫는다(좀비 크롬 방지)


# 4) 복사해 둔 html에서 시간대별 항목 추출 (BeautifulSoup) ----------
soup = BeautifulSoup(html, 'html.parser')

rows = []
for item in soup.select(ITEM):
    info = {}
    for li in item.select('li'):
        # 라벨 span은 class="hid"로 숨겨져 있지만 get_text()로는 잘 읽힌다
        text = li.get_text().replace('\xa0', ' ').strip()
        if ':' in text:
            key, value = text.split(':', 1)      # '시각: 15시' → 시각 / 15시
            info[key.strip()] = value.strip()    # '기온 :' 처럼 공백이 있어도 strip으로 정리
    if info:                                     # 값이 하나도 없는 껍데기 <ul>은 건너뛴다
        info['기온'] = info.get('기온', '').replace('℃', '')
        info['체감온도'] = info.get('체감온도', '').replace('℃', '')
        rows.append(info)

# 어떤 항목들을 읽어왔는지 확인 (페이지 구조가 바뀌면 여기서 바로 보인다)
if not rows:
    raise RuntimeError('한 건도 수집되지 않았습니다. F12로 선택자를 다시 확인하세요.')
print('읽어온 항목 :', list(rows[0].keys()))

# 5) DataFrame으로 정리하고 CSV로 저장 -----------------------------
#    columns= 로 원하는 항목만, 원하는 순서로 골라 담는다.
df = pd.DataFrame(rows, columns=COLUMNS)
print(df.head(8))
print(f'수집 완료 : {len(df)}개 시간대')

df.to_csv(FILE, index=False, encoding='utf-8-sig')
print(f'저장 완료 : {FILE}')          # ← 어느 폴더에 저장됐는지 꼭 확인!

# ------------------------------------------------------------------
# ★ 도전 1 : '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보를
#            수집해 보세요. (29개 → 56개 시간대)
#            버튼 선택자 : .tab-btn-wrap a.tab-btn  (F12로 확인)
#
# ★ 도전 2 : COLUMNS 에 '바람', '폭염영향' 을 추가해서 함께 저장해 보세요.
#            (위 '읽어온 항목' 출력에 나오는 이름을 그대로 넣으면 된다)
#
# ★ 도전 3 : 수집한 시간별 기온으로 꺾은선 그래프를 그려 보세요.
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.plot(df['시각'], pd.to_numeric(df['기온']), marker='o', color='#0E9CA0')
# plt.title('시간별 기온 예보'); plt.show()
# ------------------------------------------------------------------

### **[실습] '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보를 자동 수집하기.**

#### 프롬프트
>  앞에서 사용한 수집 코드를 참고하여 화면에서 '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보를 자동 수집하는 코드 만들어줘.

In [ ]:
# ============================================================
# '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보 수집하기
#  - 앞의 수집 코드에 "버튼 클릭" 한 단계를 더한 것이다.
#  - 3시간 간격(29개) → 1시간 간격(56개)으로 늘어난다.
# ============================================================
import os
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import StaleElementReferenceException

URL = 'https://www.weather.go.kr/w/weather/forecast/short-term.do'
FILE = os.path.abspath('시간별예보_1시간간격.csv')
COLUMNS = ['시각', '날씨', '기온', '체감온도', '강수확률', '습도']

ITEM = '#digital-forecast ul.item.s-item'                       # 시간대 한 칸
BTN_1H = '.tab-btn-wrap a.tab-btn[data-interval-hours="1"]'     # '1시간 간격' 버튼


# [함수] 예보가 다 그려졌는지 확인한다 -----------------------------
#   ※ 화면이 "다시 그려지는 중"에 요소를 읽으면
#      StaleElementReferenceException(낡은 참조 오류)이 난다.
def 예보_로딩됨(d):
    try:
        return any('시각' in (li.get_attribute('textContent') or '')
                   for li in d.find_elements(By.CSS_SELECTOR, ITEM + ' li'))
    except StaleElementReferenceException:
        return False


# [함수] 복사해 둔 html에서 시간대 칸들을 읽어 리스트로 돌려준다 ----
def read_forecast(html):
    soup = BeautifulSoup(html, 'html.parser')

    rows = []
    for item in soup.select(ITEM):
        info = {}
        for li in item.select('li'):
            # 라벨 span은 class="hid"로 숨겨져 있지만 get_text()로는 잘 읽힌다
            text = li.get_text().replace('\xa0', ' ').strip()
            if ':' in text:
                key, value = text.split(':', 1)      # '시각: 15시' → 시각 / 15시
                info[key.strip()] = value.strip()

        # ★ 기온은 따로 챙긴다 ★
        #   3시간 간격 : <span class="hid">기온 : </span>        ← ':' 가 있어서 위 반복문에 잡힌다
        #   1시간 간격 : <span class="hid">기온(체감온도) </span>  ← ':' 가 없어서 안 잡힌다!
        #   → 두 화면 모두에 있는 <span class="feel"> 에서 직접 읽으면 안전하다.
        #   1시간 간격의 feel 값은 '25℃(28℃)' 처럼 괄호가 붙으므로 앞부분만 쓴다.
        feel = item.select_one('span.feel')
        if feel:
            info['기온'] = feel.get_text().split('(')[0]

        if info:                                     # 값이 하나도 없는 껍데기 <ul>은 건너뛴다
            info['기온'] = info.get('기온', '').replace('℃', '').strip()
            info['체감온도'] = info.get('체감온도', '').replace('℃', '').strip()
            rows.append(info)
    return rows


driver = webdriver.Chrome()

try:
    driver.get(URL)

    # 1) 먼저 기본 화면(3시간 간격)이 다 그려질 때까지 기다린다 --------
    wait = WebDriverWait(driver, 30)
    wait.until(예보_로딩됨)
    before = len(driver.find_elements(By.CSS_SELECTOR, ITEM))
    print(f'3시간 간격 : {before}개 시간대')

    # 2) '1시간 간격' 버튼 클릭 ★동적 크롤링의 핵심★ ------------------
    #    execute_script로 누르면 버튼이 화면 아래쪽에 있어도 확실하게 눌린다.
    button = driver.find_element(By.CSS_SELECTOR, BTN_1H)
    driver.execute_script('arguments[0].click();', button)
    print("'1시간 간격' 버튼 클릭!")

    # 3) 표가 "다시 그려질 때까지" 기다린다 ---------------------------
    #    time.sleep(2)로 무작정 기다리지 말고,
    #    칸 수가 늘어날 때까지 기다리는 것이 훨씬 정확하다.
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, ITEM)) > before)
    wait.until(예보_로딩됨)          # 칸 안의 값까지 다 채워졌는지 한 번 더 확인
    print(f'1시간 간격 : {len(driver.find_elements(By.CSS_SELECTOR, ITEM))}개 시간대로 늘어남')

    # 4) ★ 완성된 화면을 통째로 복사해 둔다 ★ -------------------------
    #    브라우저에서 한 칸씩 읽으면 그 사이에 화면이 바뀌어 낡은 참조 오류가 난다.
    html = driver.page_source

finally:
    driver.quit()          # 오류가 나도 크롬은 반드시 닫는다(좀비 크롬 방지)


# 5) 촘촘해진 예보 읽어오기 ---------------------------------------
rows = read_forecast(html)

# 6) DataFrame으로 정리하고 CSV로 저장 -----------------------------
if not rows:
    raise RuntimeError('한 건도 수집되지 않았습니다. F12로 선택자를 다시 확인하세요.')
print('읽어온 항목 :', list(rows[0].keys()))

df = pd.DataFrame(rows, columns=COLUMNS)
print(df.head(10))
print(f'수집 완료 : {len(df)}개 시간대')

df.to_csv(FILE, index=False, encoding='utf-8-sig')
print(f'저장 완료 : {FILE}')

--------------

---------------

### [참고]  Selenium을 사용하여 동적 웹 페이지와 상호작용하기

* **(클릭 이벤트를 위한 xpath 복사)작업 순서**
    - 크롬에서 target 페이지 접속(https://www.naver.com/)
    - F12 눌러 오른쪽 영역에 개발자 페이지 나타나도록 함(html코드 나타남)
    - ctrl+shift+c 누른 상태에서 클릭 이벤트 발생할 곳 찾아 마우스 클릭
    - 해당 html코드 영역에서 마우스 오른쪽키 누르고 copy>copy.xpath 메뉴 선택하여 이벤트 코드 클립보드에 복사
    - driver.find_element(By.XPATH, '복사된 내용 붙여넣기').click()

* **[사용방법] 버튼(링크) 클릭**

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By

# 드라이버 초기화
driver = webdriver.Chrome()

# 웹페이지로 이동
driver.get('https://www.naver.com/')

# 클릭(copy.xpath 이용)  //*[@id="search-btn"]
#search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]')
#search_button.click()
search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]')
driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭

#### [1단계] 네이버 메인페이지에서 검색어 입력하고 버튼 클릭하기

In [ ]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# chrome driver를 자동으로 설치함
chromedriver_autoinstaller.install()

# 드라이버 초기화
driver = webdriver.Chrome()

def naver_main_search(driver, keyword):
    driver.get('https://www.naver.com/') # 웹페이지 로드
    search_box = driver.find_element(By.XPATH, '//*[@id="query"]')  # 검색 키워드 영역
    search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]') # 검색 버튼
    search_keyword = keyword  # 키워드
    search_box.send_keys(search_keyword)
    driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭

# 1.네이버 메인 검색
keyword = '노벨문학상'
naver_main_search(driver, keyword)
print(f'현재URL : {driver.current_url}')


#### [2단계] 네이버 검색 결과 페이지에서 다시 버튼 클릭
- 버튼 클릭 위치 확인(xPath) : 마우스오른쪽버튼 > Copy >Copy xPath

In [ ]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By

# chrome driver를 자동으로 설치함
chromedriver_autoinstaller.install()

# 드라이버 초기화
driver = webdriver.Chrome()

# [CODE 1] : 검색어 넣고 네이버 메인 검색
def naver_main_search(driver, keyword):
    print('\n1단계 : 검색어 넣고 네이버 메인 검색......')
    driver.get('https://www.naver.com/') # 웹페이지 로드
    search_box = driver.find_element(By.XPATH, '//*[@id="query"]')  # 검색 키워드 영역
    search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]') # 검색 버튼
    search_keyword = keyword  # 키워드
    search_box.send_keys(search_keyword) # 검색창에 검색어 반영
    driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭

# [CODE 2] : 검색 결과에서 다른 탭 선택
def naver_main_search_tab(driver, url, xpath=None, tab_name='뉴스'):
    print('\n2단계 : 검색 결과에서 탭 선택......')
    print(f'      currnet_url={driver.current_url}')
    driver.get(url) # 해당 웹페이지 로드
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#lnb a')))

    tab_links = driver.find_elements(By.CSS_SELECTOR, '#lnb a')
    search_button = next((link for link in tab_links if tab_name in link.text or 'where=news' in (link.get_attribute('href') or '')), None)
    if search_button is None and xpath:
        search_button = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
    if search_button is None:
        raise Exception(f'{tab_name} 탭을 찾을 수 없습니다.')
    driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'ul.list_news._infinite_list')))


keyword = input('페이지 검색어 입력: ')

# 1.[CODE 1] : 검색어 넣고 네이버 메인 검색
naver_main_search(driver, keyword)


# 2.[CODE 2] : 검색 결과에서 다른 탭 선택 ( 마우스오른쪽버튼 > Copy >Copy xPath)
naver_main_search_tab(driver, driver.current_url, '//*[@id="lnb"]/div[1]/div/div[1]/div[3]/a' )

-------------------------

#### [실습]  커피빈매장 정보 크롤링하여 파일로 저장하기
- 아래 사이트를 이용해 호출해야할 자바스크립트 함수를 확인하다.
- https://www.coffeebeankorea.com
- https://www.coffeebeankorea.com/store/store.asp
- (매장 번호로) 자세히보기: javascript:storePop2('374');
- chromedriver.exe 파일 위치는 코드와 동일한 위치에 놓는다.

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime

from selenium import webdriver
import time

MAX = 10     # 추출 데이터 건수
FILE = './CoffeeBean_매장정보.csv'

#[CODE 1]
def getStoreInfo():
    CoffeeBean_URL = "https://www.coffeebeankorea.com/store/store.asp"

    # 드라이버 초기화
    driver = webdriver.Chrome()

    result = []  # 데이터 저장 변수
    total, cnt = 370, 0
    for i in range(1, total+1):  #매장 수 만큼(370) 반복
        driver.get(CoffeeBean_URL)
        time.sleep(1)  #웹페이지 연결할 동안 1초 대기
        try:
            print(f'read[{i}]')
            driver.execute_script("storePop2(%d)" %i)
            time.sleep(1) #스크립트 실행 할 동안 1초 대기

            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            store_name_h2 = soup.select("div.store_txt > h2")
            store_name = store_name_h2[0].string  #매장 이름

            store_info = soup.select("div.store_txt > table.store_table > tbody > tr > td")
            store_address_list = list(store_info[2])
            store_address = store_address_list[0]  #매장 주소

            store_phone = store_info[3].string     #매장 전화번호
            result.append([store_name]+[store_address]+[store_phone])
            cnt += 1
            # 매장정보 가져온 데이터 출력하기
            print("save[%3d] %3d - %s" % (cnt, i, store_name))

             # MAX값에 해당하는 건수 만큼만 실행하기
            if cnt >= MAX:
                break

        except:
            continue

    return result

#---------------
# main
#---------------
#[CODE 0]
def main():
    result = []
    print('CoffeeBean store crawling >>>>>>>>>>>>>>>>>>>>>>>>>>')
    result = getStoreInfo()  #[매장 추출 함수]호출하기   #[CODE 1] 호출
    coffeebean_tbl = pd.DataFrame(result, columns=('store', 'address','phone'))
    coffeebean_tbl.to_csv(FILE, encoding='cp949', mode='w', index=True)  # 파일로 저장하기
    del result[:]
    return coffeebean_tbl


df = main() #[CODE 0] 호출
df.head()

driver.quit()


----------------------